## התאמה משוקללת (weighted least squares)

`linear_fit` מהסעיפים הקודמים מתייחסת לכל נקודה **באותה חשיבות**. אבל אנחנו כבר יודעים ש-`sigma_y` שונה מזווית לזווית (זווית 60, למשל, עם ה-SEM הגדול ביותר -- ראינו בסעיף 11.8). הגיוני שנקודה עם שגיאת מדידה קטנה תשפיע על ההתאמה **יותר** מנקודה עם שגיאה גדולה. זה בדיוק מה שהתאמה משוקללת עושה.

In [1]:
import numpy as np
import pandas as pd

g = 9.8
df = pd.read_csv("lab_measurements.csv")
df_clean = df.dropna()
angles = sorted(df_clean["angle_deg"].unique())

x = np.array([np.sin(2*np.radians(a)) for a in angles])
y = np.array([df_clean[df_clean["angle_deg"] == a]["range_measured"].mean() for a in angles])
sigma_y = np.array([
    df_clean[df_clean["angle_deg"] == a]["range_measured"].std(ddof=1) / np.sqrt(len(df_clean[df_clean["angle_deg"] == a]))
    for a in angles
])
print("sigma_y:", np.round(sigma_y, 3))

sigma_y: [0.601 0.976 1.137 2.415 0.451]


### נוסחת ההתאמה המשוקללת

המשקל של נקודה: $w_i = 1/\sigma_{y_i}^2$ -- נקודה עם שגיאה **קטנה** מקבלת משקל **גדול**. הנוסחה זהה לחלוטין לנוסחת הריבועים הפחותים הרגילה, רק שכל הסכומים הופכים לסכומים משוקללים, וממוצעים פשוטים הופכים לממוצעים משוקללים:

$$\bar x_w = \frac{\sum w_i x_i}{\sum w_i} \qquad m_w = \frac{\sum w_i (x_i-\bar x_w)(y_i-\bar y_w)}{\sum w_i (x_i-\bar x_w)^2} \qquad b_w = \bar y_w - m_w \bar x_w$$

In [2]:
def weighted_linear_fit(x, y, sigma_y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    w = 1.0 / sigma_y**2
    x_bar_w = np.sum(w*x) / np.sum(w)
    y_bar_w = np.sum(w*y) / np.sum(w)
    m_w = np.sum(w * (x - x_bar_w) * (y - y_bar_w)) / np.sum(w * (x - x_bar_w)**2)
    b_w = y_bar_w - m_w * x_bar_w
    return m_w, b_w

def linear_fit(x, y):
    x_bar, y_bar = x.mean(), y.mean()
    m = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar)**2)
    b = y_bar - m * x_bar
    return m, b

m_unweighted, b_unweighted = linear_fit(x, y)
m_weighted, b_weighted = weighted_linear_fit(x, y, sigma_y)

print(f"לא משוקלל: m = {m_unweighted:.3f}, b = {b_unweighted:.3f}")
print(f"משוקלל:    m = {m_weighted:.3f}, b = {b_weighted:.3f}")

לא משוקלל: m = 41.297, b = 0.443
משוקלל:    m = 39.691, b = 1.135


ההבדל בין שתי ההתאמות נובע כמעט לגמרי מזווית 60 -- לה יש את ה-`sigma_y` הגדול ביותר, ולכן במשוקלל היא "משפיעה פחות" מאשר בלא-משוקלל, שם כל הנקודות שוות בחשיבותן.

### באג נפוץ: משקל = sigma במקום 1/sigma²

טעות שכיחה: `w = sigma_y` (או `w = 1/sigma_y`, בלי הריבוע) במקום `w = 1/sigma_y**2`. זה **הופך** את היחסים -- נקודות עם שגיאה **גדולה** מקבלות משקל גדול יותר, בדיוק ההפך מהכוונה. שוב, אין שגיאת ריצה -- רק תוצאה הפוכה למה שרצינו.

In [3]:
def weighted_linear_fit_WRONG(x, y, sigma_y):
    w = sigma_y   # באג: משקל = sigma, לא 1/sigma^2
    x_bar_w = np.sum(w*x) / np.sum(w)
    y_bar_w = np.sum(w*y) / np.sum(w)
    m_w = np.sum(w * (x - x_bar_w) * (y - y_bar_w)) / np.sum(w * (x - x_bar_w)**2)
    return m_w

m_wrong = weighted_linear_fit_WRONG(x, y, sigma_y)
print(f"משוקלל נכון (1/sigma^2): {m_weighted:.3f}")
print(f"משוקלל שגוי (sigma):     {m_wrong:.3f}")

משוקלל נכון (1/sigma^2): 39.691
משוקלל שגוי (sigma):     41.941


### נסו בעצמכם

הראו שכאשר `sigma_y` **קבוע לכולם** (אותו ערך לכל הנקודות), ההתאמה המשוקללת שווה בדיוק להתאמה הלא-משוקללת.

In [4]:
# sigma_const = np.full_like(sigma_y, 1.0)
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
sigma_const = np.full_like(sigma_y, 1.0)
m_test, b_test = weighted_linear_fit(x, y, sigma_const)
print(np.allclose([m_test, b_test], [m_unweighted, b_unweighted]))   # True
```
זה הגיוני: כש-`sigma_y` זהה לכל הנקודות, `w=1/sigma_y**2` זהה לכל הנקודות, וממוצע משוקלל בין משקלים שווים הוא בדיוק ממוצע רגיל.
`````

### בדקו את עצמכם

In [5]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "בהתאמה משוקללת, מדוע נקודה עם sigma_y קטן מקבלת משקל גדול יותר?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "כי sigma_y קטן תמיד אומר שהערך עצמו גדול יותר", "correct": False, "feedback": "אין קשר בין גודל הערך לגודל השגיאה שלו."},
            {"answer": "אין הבדל אמיתי - זו רק קונבנציה חישובית", "correct": False, "feedback": "לא - יש לזה משמעות סטטיסטית: אמון גבוה יותר בנתונים מדויקים יותר."},
            {"answer": "כי היא נמדדה בדיוק גבוה יותר, ולכן סביר יותר שהיא קרובה לערך האמיתי", "correct": True, "feedback": "נכון."},
            {"answer": "כי המשקל בהתאמה משוקללת נקבע אך ורק לפי מספר סידורי הנקודה", "correct": False, "feedback": "לא — המשקל נקבע לפי הדיוק הסטטיסטי (sigma_y) של כל נקודה, לא לפי מיקומה ברשימה."}
        ]
    }
]
display_quiz(questions)

<IPython.core.display.Javascript object>

### תרגול עצמי

חשבו את ה-$\chi^2_\nu$ (מסעיף 11.8) עבור ההתאמה **המשוקללת**, והשוו ל-$\chi^2_\nu$ של ההתאמה הלא-משוקללת. איזו מהן נותנת $\chi^2_\nu$ קרוב יותר ל-1?

In [6]:
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
def reduced_chi2(x, y, sigma_y, m, b):
    resid = y - (m*x + b)
    chi2 = np.sum((resid/sigma_y)**2)
    return chi2 / (len(x) - 2)

rchi2_unweighted = reduced_chi2(x, y, sigma_y, m_unweighted, b_unweighted)
rchi2_weighted = reduced_chi2(x, y, sigma_y, m_weighted, b_weighted)
print(f"chi^2_nu לא-משוקלל: {rchi2_unweighted:.3f}")
print(f"chi^2_nu משוקלל:    {rchi2_weighted:.3f}")
```
`````